# Bengali ASR — going for 90%+ word accuracy (WER < 10%)

## The plan (and why)

| Choice | What we use | Why |
|---|---|---|
| **Dataset** | **OpenSLR SLR53** — 229 hrs, ~196k utterances of **clean, READ Bengali** | This is the key. Our old data (Vaani) was *spontaneous rural conversation* — the hardest kind (papers only get 24–31% WER on it). **Clean read speech is where <10% WER is actually reachable.** |
| **Model** | `whisper-large-v3` (fallback: `whisper-medium`) | Bigger model = lower WER. Large-v3 is the strongest multilingual ASR available. |
| **Epochs** | **3–6, NOT 110** | With a *large* dataset you need FEW epochs. 110 epochs would massively overfit and take weeks. More data + few epochs ≫ little data + many epochs. |
| **Method** | LoRA fine-tune, bf16/8-bit | Fits Colab GPU, near full-fine-tune quality |
| **Safety** | Checkpoints → **Google Drive**, auto-resume | Colab disconnects. This makes that harmless. |

## Honest expectation
- On clean read speech, a fine-tuned Whisper-large typically lands **10–20% WER** → **80–90% word accuracy**.
- **<10% WER (90%+ word accuracy) is possible here but NOT guaranteed** — it depends on how much data we get through.
- **Character accuracy will almost certainly exceed 90%** (it's the metric Bengali papers headline).
- We will report BOTH, measured on held-out clips the model never trained on.

## How to run
1. **Runtime → Change runtime type → GPU** (A100/L4 if you have Colab Pro; T4 works but is slower).
2. **Runtime → Run all.**
3. If Colab disconnects: just **Run all again** — it resumes from the last checkpoint in your Drive.

In [ ]:
# 1. Check the GPU we got
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 2. Install dependencies
!pip install -q --upgrade transformers datasets accelerate peft bitsandbytes jiwer soundfile librosa evaluate

# Colab ships an OLD torchao (0.10) that newer peft rejects with:
#   ImportError: Found an incompatible version of torchao ...
# We don't use torchao at all -> just remove it. peft then skips it cleanly.
!pip uninstall -y -q torchao

print("\ndeps ready (torchao removed to avoid the peft version clash)")

In [ ]:
# 3. Mount Google Drive -> checkpoints survive Colab disconnects (CRITICAL)
from google.colab import drive
drive.mount('/content/drive')

import os
CKPT_DIR = '/content/drive/MyDrive/bengali_asr_ckpt'
os.makedirs(CKPT_DIR, exist_ok=True)
print('checkpoints ->', CKPT_DIR)

In [ ]:
# 4. AUTO-CONFIG — tuned for the best accuracy inside a ~10 hour budget on YOUR GPU
import torch

name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
FAST = any(k in name for k in ["A100", "H100", "L4", "L40", "V100"])
print(f"GPU: {name}  ({vram:.0f} GB)  ->  {'FAST (Colab Pro)' if FAST else 'T4 (free tier)'}")

# whisper-large-v3 on BOTH tiers: its Bengali is far stronger than medium, and the
# model matters more for word accuracy than extra epochs do.
MODEL = "openai/whisper-large-v3"

if FAST:
    BATCH, GRAD_ACC, N_PARTS, EPOCHS = 8, 2, 8, 4     # ~8-10h, lots of data
else:
    BATCH, GRAD_ACC, N_PARTS, EPOCHS = 4, 4, 4, 3     # T4: batch 4 fits 16GB, ~10h

LR       = 1e-3    # standard for LoRA on Whisper
N_EVAL   = 500     # held-out clips for the honest WER/CER measurement
MAX_SEC  = 20      # skip clips longer than this

print(f"\nCONFIG -> {MODEL}")
print(f"  data parts = {N_PARTS} (~{N_PARTS*19}k clips) | epochs = {EPOCHS} | batch = {BATCH}x{GRAD_ACC}")
print("\n  If you hit CUDA OOM: set BATCH=2, GRAD_ACC=8 and re-run from this cell.")
print("  To push accuracy higher later: raise N_PARTS (max 10 = the full 229 hours).")

In [ ]:
# 5. Download OpenSLR SLR53 (clean READ Bengali speech, ~229 hrs)
#    Public, no auth needed. Each part is ~1.5GB.

BASE = "https://www.openslr.org/resources/53"

# transcripts:  utt_id \t speaker_id \t text
!wget -q -nc {BASE}/utt_spk_text.tsv

import os
for i in range(N_PARTS):
    z = f"asr_bengali_{i}.zip"
    if not os.path.exists(z):
        print(f"downloading {z} ...")
        !wget -q {BASE}/{z}
    print(f"unzipping {z} ...")
    !unzip -q -n {z} -d /content/slr53

!du -sh /content/slr53 2>/dev/null | tail -1

In [ ]:
# 6. Build the (audio_path, text) pairs
import glob, csv, random

# map utt_id -> text
utt2text = {}
with open("utt_spk_text.tsv", encoding="utf-8") as f:
    for line in f:
        p = line.rstrip("\n").split("\t")
        if len(p) >= 3:
            utt2text[p[0]] = p[2].strip()

# find the downloaded audio files
files = glob.glob("/content/slr53/**/*.flac", recursive=True)
print(f"found {len(files)} audio files, {len(utt2text)} transcripts")

pairs = []
for fp in files:
    utt = os.path.basename(fp).replace(".flac", "")
    t = utt2text.get(utt)
    if t:
        pairs.append({"path": fp, "text": t})

random.seed(0); random.shuffle(pairs)
eval_pairs, train_pairs = pairs[:N_EVAL], pairs[N_EVAL:]
print(f"TRAIN {len(train_pairs)}  |  HELD-OUT {len(eval_pairs)} (never trained on)")

In [ ]:
# 7. Dataset: load audio, make log-mel features + token labels (lazy, so RAM stays small)
import torch, librosa, numpy as np
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained(MODEL, language="bengali", task="transcribe")

class SLR53(torch.utils.data.Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, i):
        p = self.pairs[i]
        wav, _ = librosa.load(p["path"], sr=16000)
        wav = wav[: MAX_SEC * 16000].astype(np.float32)
        feat = processor.feature_extractor(wav, sampling_rate=16000).input_features[0]
        labels = processor.tokenizer(p["text"]).input_ids[:448]   # whisper decoder limit
        return {"input_features": feat, "labels": labels}

train_ds, eval_ds = SLR53(train_pairs), SLR53(eval_pairs)
print("datasets ready")

In [ ]:
# 8. Collator + metrics (WER = word accuracy, CER = character accuracy)
from dataclasses import dataclass
import jiwer

@dataclass
class Collator:
    processor: object
    def __call__(self, features):
        batch = self.processor.feature_extractor.pad(
            [{"input_features": f["input_features"]} for f in features], return_tensors="pt")
        labs = self.processor.tokenizer.pad(
            [{"input_ids": f["labels"]} for f in features], return_tensors="pt")
        batch["labels"] = labs["input_ids"].masked_fill(labs.attention_mask.ne(1), -100)
        return batch

def compute_metrics(pred):
    pid, lid = pred.predictions, pred.label_ids
    lid = np.where(lid == -100, processor.tokenizer.pad_token_id, lid)
    hyp = processor.batch_decode(pid, skip_special_tokens=True)
    ref = processor.batch_decode(lid, skip_special_tokens=True)
    pairs = [(r, h) for r, h in zip(ref, hyp) if r.strip()]
    if not pairs:
        return {"wer": 100.0, "cer": 100.0}
    r, h = zip(*pairs)
    return {"wer": 100 * jiwer.wer(list(r), list(h)),
            "cer": 100 * jiwer.cer(list(r), list(h))}

In [ ]:
# 9. Load Whisper + attach LoRA
from transformers import WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model

model = WhisperForConditionalGeneration.from_pretrained(MODEL, torch_dtype=torch.bfloat16).to("cuda")
model.generation_config.language = "bengali"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.config.use_cache = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

model = get_peft_model(model, LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.05, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]))
model.print_trainable_parameters()

In [ ]:
# 10. TRAIN  (checkpoints -> Drive, auto-resumes if Colab disconnects / GPU quota runs out)
import glob as _glob
from pathlib import Path
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

args = Seq2SeqTrainingArguments(
    output_dir=CKPT_DIR,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LR,
    warmup_steps=200,
    num_train_epochs=EPOCHS,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="steps",
    save_steps=200,           # save to Drive every 200 steps -> a disconnect costs very little
    save_total_limit=2,       # keep the 2 newest (Drive space)
    predict_with_generate=True,
    generation_max_length=225,
    logging_steps=50,
    report_to="none",
    remove_unused_columns=False,
)

# AUTO-RESUME from the newest VALID checkpoint in Drive.
# If the runtime died / GPU quota ran out, just re-run cells 1-10 and it continues here.
cks = [c for c in _glob.glob(f"{CKPT_DIR}/checkpoint-*")
       if os.path.exists(os.path.join(c, "trainer_state.json"))]   # skip half-written ones
cks.sort(key=lambda p: int(p.rsplit("-", 1)[-1]))
resume = cks[-1] if cks else None
print("resuming from:", resume if resume else "scratch (no checkpoint found)")

trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=eval_ds,
    data_collator=Collator(processor),
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)
trainer.train(resume_from_checkpoint=resume)

In [ ]:
# 11. FINAL HONEST EVAL on the held-out clips (never trained on)
model.eval()
refs, hyps = [], []
for i in range(len(eval_ds)):
    p = eval_pairs[i]
    wav, _ = librosa.load(p["path"], sr=16000)
    feat = processor.feature_extractor(wav[: MAX_SEC * 16000], sampling_rate=16000,
                                       return_tensors="pt").input_features.to("cuda").to(torch.bfloat16)
    with torch.no_grad():
        ids = model.generate(feat, max_new_tokens=225)
    hyps.append(processor.batch_decode(ids, skip_special_tokens=True)[0].strip())
    refs.append(p["text"])
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(eval_ds)}")

wer = 100 * jiwer.wer(refs, hyps)
cer = 100 * jiwer.cer(refs, hyps)
print("\n" + "=" * 60)
print(f"  HELD-OUT ({len(refs)} unseen clips, OpenSLR SLR53)")
print(f"  WER {wer:5.2f}%   ->  WORD accuracy      {100-wer:5.2f}%")
print(f"  CER {cer:5.2f}%   ->  CHARACTER accuracy {100-cer:5.2f}%")
print("=" * 60)

In [ ]:
# 12. Save the finished model to Drive
SAVE = '/content/drive/MyDrive/bengali_asr_final'
model.save_pretrained(SAVE)
processor.save_pretrained(SAVE)
print('saved ->', SAVE)

## If you don't hit 90% word accuracy — the levers, in order of impact

1. **More data** → raise `N_PARTS` (up to 10 = the full 229 hrs). This is the biggest lever by far.
2. **More epochs** → raise `EPOCHS` to 6–8 (only if the WER is still dropping each epoch — check the log).
3. **Bigger model** → make sure `MODEL = "openai/whisper-large-v3"` (not medium).
4. **Colab Pro** (A100/L4) → lets you use more data + the large model without timing out.

**Do NOT just raise epochs to 110.** Watch the per-epoch `eval_wer` in the log: once it stops dropping, more epochs only overfit (train WER falls, held-out WER rises) and the honest number gets *worse*.